In [1]:
use_powr = "STIS" # "All", "STIS", "COS", "None"

In [2]:
import os
import re
import sys
import glob
import toml
import shutil
import textwrap
import subprocess

import numpy as np
import pandas as pd

from pathlib import Path
from astropy.io import fits
from importlib import reload
from astropy.table import Table
from scipy.interpolate import interp1d
from joebvp import cfg, utils, VPmeasure

read_sets: Using set file -- 
  /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/linetools/lists/sets/llist_v1.3.ascii
Loading abundances from Asplund2009
Abundances are relative by number on a logarithmic scale with H=12
linetools.lists.parse: Reading linelist --- 
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/linetools/data/lines/morton03_table2.fits.gz
linetools.lists.parse: Reading linelist --- 
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/linetools/data/lines/verner96_tab1.fits.gz
joebvp.makevoigt: No local joebvp_cfg.py found, using default cfg.py file from joebvp.
joebvp.VPmeasure: No local joebvp_cfg.py found, using default cfg.py file from joebvp.
Loading abundances from Asplund2009
Abundances are relative by number on a logarithmic scale with H=12
linetools.lists.parse: Reading linelist --- 
   /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-package

In [3]:
target_directory = Path().resolve()
spectra_directories = sorted([d for d in os.listdir(target_directory) if d.startswith("G") or d.startswith("E")])
galaxy = target_directory.parts[target_directory.parts.index('float-for-morrow') + 1]
print(spectra_directories)
print(galaxy)
python_cmd = sys.executable
if galaxy == "LMC":
    model_file = "/Users/billyli/Documents/stellar template fitting/powr-OB-lmc-vd3.nc"
elif galaxy == "SMC":
    model_file = "/Users/billyli/Documents/stellar template fitting/powr-OB-smc-vd3.nc"
else:
    use_powr = "None"
runner_path = "/Users/billyli/Documents/stellar template fitting/stellar_template_fitting_runner.py"

['E140M_1425_0.2x0.2_2240.000_14675', 'E140M_1425_0.2x0.2_2750.000_14675', 'E140M_1425_0.2x0.2_2778.000_14675', 'E230M_1978_0.2x0.2_2240.000_14675', 'E230M_1978_0.2x0.2_2730.000_14675']
LMC


In [4]:
def parse_directory(dirname):

    Target = target_directory.name

    parts = dirname.split("_")

    Grating = parts[0]
    Cenwave = int(parts[1])
    T_exp = str(parts[3])
    PID = int(parts[4])

    if Grating.startswith("G"):
        
        Instrument = "COS"
        LP = parts[2]

        return {
            "Target": Target, 
            "Instrument": Instrument, 
            "Grating": Grating, 
            "Cenwave": Cenwave, 
            "LP": LP, 
            "Aperture": None, 
            "T_exp": T_exp, 
            "PID": PID, 
        }
        
    else:
        
        Instrument = "STIS"
        Aperture = parts[2]
        return {
            "Target": Target, 
            "Instrument": Instrument, 
            "Grating": Grating, 
            "Cenwave": Cenwave, 
            "LP": None, 
            "Aperture": Aperture, 
            "T_exp": T_exp, 
            "PID": PID, 
        }

records = [parse_directory(d) for d in spectra_directories]
df = pd.DataFrame.from_records(records)
display(df)

,Target,Instrument,Grating,Cenwave,LP,Aperture,T_exp,PID
0,BI 42,STIS,E140M,1425,None,0.2x0.2,2240.000,14675
1,BI 42,STIS,E140M,1425,None,0.2x0.2,2750.000,14675
2,BI 42,STIS,E140M,1425,None,0.2x0.2,2778.000,14675
3,BI 42,STIS,E230M,1978,None,0.2x0.2,2240.000,14675
4,BI 42,STIS,E230M,1978,None,0.2x0.2,2730.000,14675


In [5]:
if galaxy == "LMC":
    guess_redshift = 0.00087
elif galaxy == "SMC":
    guess_redshift = 0.00053

def generate_command_continuumfit_cos(path, dataset_name):
    segment = f"""cd "[path]" && lt_continuumfit --redshift {guess_redshift} [dataset_name]_x1dsum.fits [dataset_name]_continuumfit_manual.fits"""
    modified_segment = segment.replace("[dataset_name]", dataset_name)
    modified_segment = modified_segment.replace("[path]", path)
    print(modified_segment)

def generate_command_continuumfit_stis(path, dataset_name):
    segment = f"""cd "[path]" && lt_continuumfit --redshift {guess_redshift} [dataset_name]_x1d.fits [dataset_name]_continuumfit_manual.fits"""
    modified_segment = segment.replace("[dataset_name]", dataset_name)
    modified_segment = modified_segment.replace("[path]", path)
    print(modified_segment)
    
for directory in spectra_directories:
    path = target_directory / directory
    for file in os.listdir(path):
        if file.endswith("_x1dsum.fits"):
            dataset_name = file.replace("_x1dsum.fits", "")
            path = str(path) + "/"
            generate_command_continuumfit_cos(path, dataset_name)
            print("\r")
        elif file.endswith("_x1d.fits"):
            dataset_name = file.replace("_x1d.fits", "")
            path = str(path) + "/"
            generate_command_continuumfit_stis(path, dataset_name)
            print("\r")

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2240.000_14675/" && lt_continuumfit --redshift 0.00087 oda973010_x1d.fits oda973010_continuumfit_manual.fits

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2750.000_14675/" && lt_continuumfit --redshift 0.00087 oda973030_x1d.fits oda973030_continuumfit_manual.fits

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2778.000_14675/" && lt_continuumfit --redshift 0.00087 oda973020_x1d.fits oda973020_continuumfit_manual.fits

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E230M_1978_0.2x0.2_2240.000_14675/" && lt_continuumfit --redshift 0.00087 oda974010_x1d.fits oda974010_continuumfit_manual.fits

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E230M_1978_0.2x0.2_2730.000_14675/" && lt_continuumfit --redshift 0.00087 oda974020_x1d.fits oda974020_continuumfit_manual.fits



In [ ]:
for i in range(len(spectra_directories)):

    directory = spectra_directories[i]
    path = target_directory / directory
    Instrument = df.iloc[i]["Instrument"]

    if Instrument == "COS" and (use_powr == "All" or use_powr == "COS"):

        for file in os.listdir(path):

            Grating = df.iloc[i]["Grating"]
            
            if file.endswith("_continuumfit_manual.fits") and Grating == "G130M":

                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"

                problem = df.iloc[i]["Target"].lower()
                target = df.iloc[i]["Target"].upper()
            
                spec_file = src
                
                wave_min = 1132
                wave_max = 1420
            
                toml_content = textwrap.dedent(f"""
                [target_info]
                problem    = "{problem}"
                target     = "{target}"
                spec_file  = "{spec_file}"
                model_file = "{model_file}"
                instrument = "cos_g130m"
            
                [include_wave_range]
                wave_range = [ {wave_min}, {wave_max},]
            
                [exclude_wave_ranges]
                dla = [[1200, 1220,] ]
                gap = [[1270, 1290,] ]
            
                [vshift]
                min = 50
                max = 550
                delta = 50
            
                [vsini]
                min = 50
                max = 350
                delta = 50
            
                [exclude_vranges.ions]
                linelist = "ISM"
                vrange = [-75, 350]
                drop_species = []
            
                [target_info.plot_ranges_kwargs.all_merged]
                color = "gainsboro"
                """).lstrip()

                tmp_toml = subdir / f"{problem}.toml"
                tmp_toml.write_text(toml_content)
            
                subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = subdir)
            
                csv_path = subdir / f"{problem}.csv"
                model_continuum = Table.read(str(csv_path), format = "csv")
            
                wave = model_continuum['wave'].data
                flux = model_continuum['flux'].data
                err = model_continuum['err'].data
                wave_model = model_continuum['wave'].data
                best_fit_spec = model_continuum['best_fit_spec'].data
            
                with fits.open(src, mode = 'readonly') as hdul:
                    new_hdul = fits.HDUList([h.copy() for h in hdul])
            
                flux_original = new_hdul[0].data
                wave_original = new_hdul[2].data
                continuum_hdu = new_hdul[3]
                
                continuum_original = continuum_hdu.data
                
                mask = (wave_original >= wave_min) & (wave_original <= wave_max)
                flux_original_masked = flux_original[mask]
            
                normalization_factor = flux_original_masked[1000] / flux[1000]
                normalization_factor_2 = flux_original_masked[1500] / flux[1500]
                normalization_factor_3 = flux_original_masked[2000] / flux[2000]
            
                if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
                    raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
            
                continuum_hdu.data[mask] = best_fit_spec * normalization_factor
            
                new_hdul.writeto(str(dst), overwrite = True)

            if file.endswith("_continuumfit_manual.fits") and Grating == "G160M":

                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"

                problem = df.iloc[i]["Target"].lower()
                target = df.iloc[i]["Target"].upper()
            
                spec_file = src
                
                wave_min = 1400
                wave_max = 1800
            
                toml_content = textwrap.dedent(f"""
                [target_info]
                problem    = "{problem}"
                target     = "{target}"
                spec_file  = "{spec_file}"
                model_file = "{model_file}"
                instrument = "cos_g160m"
            
                [include_wave_range]
                wave_range = [ {wave_min}, {wave_max},]
            
                [exclude_wave_ranges]
            
                [vshift]
                min = 50
                max = 550
                delta = 50
            
                [vsini]
                min = 50
                max = 350
                delta = 50
            
                [exclude_vranges.ions]
                linelist = "ISM"
                vrange = [-75, 350]
                drop_species = []
            
                [target_info.plot_ranges_kwargs.all_merged]
                color = "gainsboro"
                """).lstrip()

                tmp_toml = subdir / f"{problem}.toml"
                tmp_toml.write_text(toml_content)
            
                subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = subdir)
            
                csv_path = subdir / f"{problem}.csv"
                model_continuum = Table.read(str(csv_path), format = "csv")
            
                wave = model_continuum['wave'].data
                flux = model_continuum['flux'].data
                err = model_continuum['err'].data
                wave_model = model_continuum['wave'].data
                best_fit_spec = model_continuum['best_fit_spec'].data
            
                with fits.open(src, mode = 'readonly') as hdul:
                    new_hdul = fits.HDUList([h.copy() for h in hdul])
            
                flux_original = new_hdul[0].data
                wave_original = new_hdul[2].data
                continuum_hdu = new_hdul[3]
                
                continuum_original = continuum_hdu.data
                
                mask = (wave_original >= wave_min) & (wave_original <= wave_max)
                flux_original_masked = flux_original[mask]
            
                normalization_factor = flux_original_masked[1000] / flux[1000]
                normalization_factor_2 = flux_original_masked[1500] / flux[1500]
                normalization_factor_3 = flux_original_masked[2000] / flux[2000]
            
                if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
                    raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
            
                continuum_hdu.data[mask] = best_fit_spec * normalization_factor
            
                new_hdul.writeto(str(dst), overwrite = True)

            if file.endswith("_continuumfit_manual.fits") and Grating == "G185M":

                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"
                
                problem = df.iloc[i]["Target"].lower()
                target = df.iloc[i]["Target"].upper()
            
                spec_file = src
                
                wave_min = 1800
                wave_max = 2100
            
                toml_content = textwrap.dedent(f"""
                [target_info]
                problem    = "{problem}"
                target     = "{target}"
                spec_file  = "{spec_file}"
                model_file = "{model_file}"
                instrument = "cos_g185m"
            
                [include_wave_range]
                wave_range = [ {wave_min}, {wave_max},]
            
                [exclude_wave_ranges]
                dla = [[1200, 1220,] ]
                gap = [[1270, 1290,] ]
            
                [vshift]
                min = 50
                max = 550
                delta = 50
            
                [vsini]
                min = 50
                max = 350
                delta = 50
            
                [exclude_vranges.ions]
                linelist = "ISM"
                vrange = [-75, 350]
                drop_species = []
            
                [target_info.plot_ranges_kwargs.all_merged]
                color = "gainsboro"
                """).lstrip()

                tmp_toml = subdir / f"{problem}.toml"
                tmp_toml.write_text(toml_content)
            
                subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = subdir)
            
                csv_path = subdir / f"{problem}.csv"
                model_continuum = Table.read(str(csv_path), format = "csv")
            
                wave = model_continuum['wave'].data
                flux = model_continuum['flux'].data
                err = model_continuum['err'].data
                wave_model = model_continuum['wave'].data
                best_fit_spec = model_continuum['best_fit_spec'].data
            
                with fits.open(src, mode = 'readonly') as hdul:
                    new_hdul = fits.HDUList([h.copy() for h in hdul])
            
                flux_original = new_hdul[0].data
                wave_original = new_hdul[2].data
                continuum_hdu = new_hdul[3]
                
                continuum_original = continuum_hdu.data
                
                mask = (wave_original >= wave_min) & (wave_original <= wave_max)
                flux_original_masked = flux_original[mask]
            
                normalization_factor = flux_original_masked[1000] / flux[1000]
                normalization_factor_2 = flux_original_masked[1500] / flux[1500]
                normalization_factor_3 = flux_original_masked[2000] / flux[2000]
            
                if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
                    raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
            
                continuum_hdu.data[mask] = best_fit_spec * normalization_factor
            
                new_hdul.writeto(str(dst), overwrite = True)
                
    if Instrument == "STIS" and (use_powr == "All" or use_powr == "STIS"):

        for file in os.listdir(path):

            Grating = df.iloc[i]["Grating"]
            
            if file.endswith("_continuumfit_manual.fits") and Grating == "E140M":

                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"

                problem = df.iloc[i]["Target"].lower()
                target = df.iloc[i]["Target"].upper()
            
                spec_file = src
                
                wave_min = 1250
                wave_max = 1650
            
                toml_content = textwrap.dedent(f"""
                [target_info]
                problem    = "{problem}"
                target     = "{target}"
                spec_file  = "{spec_file}"
                model_file = "{model_file}"
                instrument = "stis_e140m"
            
                [include_wave_range]
                wave_range = [ {wave_min}, {wave_max},]
            
                [exclude_wave_ranges]
            
                [vshift]
                min = 50
                max = 550
                delta = 50
            
                [vsini]
                min = 50
                max = 350
                delta = 50
            
                [exclude_vranges.ions]
                linelist = "ISM"
                vrange = [-75, 350]
                drop_species = []
            
                [target_info.plot_ranges_kwargs.all_merged]
                color = "gainsboro"
                """).lstrip()

                tmp_toml = subdir / f"{problem}.toml"
                tmp_toml.write_text(toml_content)
            
                subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = subdir)
            
                csv_path = subdir / f"{problem}.csv"
                model_continuum = Table.read(str(csv_path), format = "csv")
            
                wave = model_continuum['wave'].data
                flux = model_continuum['flux'].data
                err = model_continuum['err'].data
                wave_model = model_continuum['wave'].data
                best_fit_spec = model_continuum['best_fit_spec'].data
            
                with fits.open(src, mode = 'readonly') as hdul:
                    new_hdul = fits.HDUList([h.copy() for h in hdul])
            
                flux_original = new_hdul[0].data
                wave_original = new_hdul[2].data
                continuum_hdu = new_hdul[3]
                
                continuum_original = continuum_hdu.data
                
                mask = (wave_original >= wave_min) & (wave_original <= wave_max)
                flux_original_masked = flux_original[mask]
            
                normalization_factor = flux_original_masked[1000] / flux[1000]
                normalization_factor_2 = flux_original_masked[1500] / flux[1500]
                normalization_factor_3 = flux_original_masked[2000] / flux[2000]
            
                if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
                    raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
            
                continuum_hdu.data[mask] = best_fit_spec * normalization_factor
            
                new_hdul.writeto(str(dst), overwrite = True)

            if file.endswith("_continuumfit_manual.fits") and Grating == "E230M":

                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"

                problem = df.iloc[i]["Target"].lower()
                target = df.iloc[i]["Target"].upper()
            
                spec_file = src
                
                wave_min = 1600
                wave_max = 3100
            
                toml_content = textwrap.dedent(f"""
                [target_info]
                problem    = "{problem}"
                target     = "{target}"
                spec_file  = "{spec_file}"
                model_file = "{model_file}"
                instrument = "stis_e230m"
            
                [include_wave_range]
                wave_range = [ {wave_min}, {wave_max},]
            
                [exclude_wave_ranges]
            
                [vshift]
                min = 50
                max = 550
                delta = 50
            
                [vsini]
                min = 50
                max = 350
                delta = 50
            
                [exclude_vranges.ions]
                linelist = "ISM"
                vrange = [-75, 350]
                drop_species = []
            
                [target_info.plot_ranges_kwargs.all_merged]
                color = "gainsboro"
                """).lstrip()

                tmp_toml = subdir / f"{problem}.toml"
                tmp_toml.write_text(toml_content)
            
                subprocess.run([python_cmd, runner_path, str(tmp_toml)], check = True, cwd = subdir)
            
                csv_path = subdir / f"{problem}.csv"
                model_continuum = Table.read(str(csv_path), format = "csv")
            
                wave = model_continuum['wave'].data
                flux = model_continuum['flux'].data
                err = model_continuum['err'].data
                wave_model = model_continuum['wave'].data
                best_fit_spec = model_continuum['best_fit_spec'].data
            
                with fits.open(src, mode = 'readonly') as hdul:
                    new_hdul = fits.HDUList([h.copy() for h in hdul])
            
                flux_original = new_hdul[0].data
                wave_original = new_hdul[2].data
                continuum_hdu = new_hdul[3]
                
                continuum_original = continuum_hdu.data
                
                mask = (wave_original >= wave_min) & (wave_original <= wave_max)
                flux_original_masked = flux_original[mask]
            
                normalization_factor = flux_original_masked[1000] / flux[1000]
                normalization_factor_2 = flux_original_masked[1500] / flux[1500]
                normalization_factor_3 = flux_original_masked[2000] / flux[2000]
            
                if not (np.allclose([normalization_factor, normalization_factor_2, normalization_factor_3], normalization_factor, rtol = 0.001)):
                    raise ValueError(f"Normalization factors disagree: {normalization_factor:.3f}, {normalization_factor_2:.3f}, {normalization_factor_3:.3f}")
            
                continuum_hdu.data[mask] = best_fit_spec * normalization_factor
            
                new_hdul.writeto(str(dst), overwrite = True)

    else: 

        for file in os.listdir(path):
            if file.endswith("_continuumfit_manual.fits"):
                
                dataset_name = file.replace("_continuumfit_manual.fits", "")
                subdir = path / dataset_name
                subdir.mkdir(exist_ok = True)
                src = path / file
                dst = subdir / f"{dataset_name}_continuumfit.fits"
                shutil.copy(src, dst)

In [6]:
def generate_command_igmguesses_cos(path, dataset_name):
    segment = f"""cd "[path]" && pyigm_igmguesses [dataset_name]_continuumfit.fits -o [dataset_name]_x1dfits_model.json"""
    modified_segment = segment.replace("[dataset_name]", dataset_name)
    modified_segment = modified_segment.replace("[path]", path)
    print(modified_segment)

def generate_command_igmguesses_stis(path, dataset_name):
    segment = f"""cd "[path]" && pyigm_igmguesses [dataset_name]_continuumfit.fits -o [dataset_name]_x1dfits_model.json"""
    modified_segment = segment.replace("[dataset_name]", dataset_name)
    modified_segment = modified_segment.replace("[path]", path)
    print(modified_segment)


target_directory = Path().resolve()
spectra_directories = sorted([d for d in os.listdir(target_directory) if d.startswith("G") or d.startswith("E")])

for directory in spectra_directories:
    path = target_directory / directory
    for subdir in os.listdir(path):
        subdir_path = path / subdir
        if not subdir_path.is_dir():
            continue
        for file in os.listdir(subdir_path):
            file_path = subdir_path / file
            if file.endswith("_continuumfit.fits"):
                dataset_name = file.replace("_continuumfit.fits", "")
                generate_command_igmguesses_cos(str(subdir_path) + "/", dataset_name)
                print("\r")

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2240.000_14675/oda973010/" && pyigm_igmguesses oda973010_continuumfit.fits -o oda973010_x1dfits_model.json

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2750.000_14675/oda973030/" && pyigm_igmguesses oda973030_continuumfit.fits -o oda973030_x1dfits_model.json

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2778.000_14675/oda973020/" && pyigm_igmguesses oda973020_continuumfit.fits -o oda973020_x1dfits_model.json

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E230M_1978_0.2x0.2_2240.000_14675/oda974010/" && pyigm_igmguesses oda974010_continuumfit.fits -o oda974010_x1dfits_model.json

cd "/Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E230M_1978_0.2x0.2_2730.000_14675/oda974020/" && pyigm_igmguesses oda974020_continuumfit.fits -o oda974020_x1dfits_model.json



In [9]:
for i in range(len(spectra_directories)):

    directory = spectra_directories[i]
    path = target_directory / directory

    os.chdir(path)
    
    Instrument = df.iloc[i]["Instrument"]
    Grating = df.iloc[i]["Grating"]
    Cenwave = str(df.iloc[i]["Cenwave"])
    if Instrument == "COS":
        LP = str(df.iloc[i]["LP"].lstrip("LP"))
        Aperture = "NA"
    elif Instrument == "STIS":
        LP = "NA"
        Aperture = df.iloc[i]["Aperture"]

    if Grating == "G130M":
        LSF_Ranges = [1100, 1460]
    elif Grating == "G160M":
        LSF_Ranges = [1400, 1800]
    elif Grating == "G185M":
        LSF_Ranges = [1800, 2100]
    elif Grating == "E140M":
        LSF_Ranges = [1144, 1729]
    elif Grating == "E230M":
        LSF_Ranges = [1607, 3129]

    for subdir in os.listdir(path):
        
        subdir_path = path / subdir
        if not subdir_path.is_dir():
            continue
        os.chdir(subdir_path)
    
        igm_model = glob.glob(os.path.join(subdir_path, "*_x1dfits_model.json"))[0]
        continuumfit = glob.glob(os.path.join(subdir_path, "*_continuumfit.fits"))[0]
    
        utils.pyigm_to_veeper(igm_model, continuumfit)

        component_groups_dir = os.path.join(subdir_path, "component_groups")
        output_dir = os.path.join(subdir_path, "modified_component_groups")
    
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
    
        mc_lines_file = os.path.join(str(target_directory.parent.parent), f"{Grating}.txt")
        mc_lines_df = pd.read_csv(mc_lines_file, sep = r"\s+", header = None, names = ["line", "wavelength"])
        mc_lines_df["wavelength"] = mc_lines_df["wavelength"].astype(float)
    
        def is_valid_line(row, line_df):
            trans = row["trans"].strip()
            restwave = row["restwave"]
            matching_lines = line_df[line_df["line"] == trans]
            for wave in matching_lines["wavelength"]:
                if wave <= restwave <= 1.001 * wave:
                    return True
            return False
    
        input_files = glob.glob(os.path.join(component_groups_dir, "*.txt"))
    
        for file in input_files:
            df_component = pd.read_csv(file, sep = "|")
            df_component = df_component.loc[:, ~df_component.columns.str.contains('^Unnamed')]
        
            df_component["restwave"] = pd.to_numeric(df_component["restwave"], errors = "coerce")
            df_component["bval"] = pd.to_numeric(df_component["bval"], errors = "coerce")
        
            valid_mask = df_component.apply(lambda row: is_valid_line(row, mc_lines_df), axis = 1)
            df_component_filtered = df_component[valid_mask].copy()
        
            df_component_filtered["bval"] = df_component_filtered["bval"]
        
            output_file = os.path.join(output_dir, os.path.basename(file))
            df_component_filtered.to_csv(output_file, sep = "|", index = False)
    
        input_group_files = sorted([f for f in glob.glob(os.path.join(output_dir, "input_group_*.txt")) if re.match(r".*input_group_\d+\.txt$", f)])
        
        reload(cfg)
        cfg.lsfs = []
        if Instrument == "COS":
            setattr(cfg, 'lps', [LP])
        elif Instrument == "STIS":
            setattr(cfg, 'slits', [Aperture])
        for entry, col in [[Instrument, 'instr'], [Grating, 'gratings'], [Cenwave, 'cen_wave'], [None, 'pixel_scales'], [None, 'fwhms']]:
            setattr(cfg, col, [entry])
        cfg.lsfranges = np.array([LSF_Ranges])
        
        VPmeasure.batch_fit(continuumfit, input_group_files, filepath = './modified_component_groups/')

Loading abundances from Asplund2009
Abundances are relative by number on a logarithmic scale with H=12
Bad METADATA;  proceeding without
`ftol` termination condition is satisfied.
Function evaluations 12, initial cost 2.3506e+02, final cost 3.2672e+01, first-order optimality 3.12e-03.

Fit results: 

1608.45	 0.000890	 15.391	 20.000	 3.9929
 	  	  	 0.101	 1.126	 1.003 

1611.2	 0.000890	 15.391	 20.000	 3.9929
 	  	  	 0.0	 0.0	 0.0 


Reduced chi-squared: 0.947027
Iteration 1 -
`ftol` termination condition is satisfied.
Function evaluations 12, initial cost 2.3506e+02, final cost 3.2672e+01, first-order optimality 3.12e-03.

Fit results: 

1608.45	 0.000890	 15.391	 20.000	 3.9929
 	  	  	 0.101	 1.126	 1.003 

1611.2	 0.000890	 15.391	 20.000	 3.9929
 	  	  	 0.0	 0.0	 0.0 


Reduced chi-squared: 0.947027
Iteration 2 -
Fit converged after 2 iterations.
VPmeasure: Fit converged: /Users/billyli/Documents/float-for-morrow/LMC/LMC N11B/BI 42/E140M_1425_0.2x0.2_2240.000_14675/oda973010/

In [10]:
records = []

for i in range(len(spectra_directories)):

    directory = spectra_directories[i]
    path = target_directory / directory
    
    for subdir in os.listdir(path):
        
        subdir_path = path / subdir
        if not subdir_path.is_dir():
            continue
        fn = os.path.join(subdir_path, "compiledVPoutputs.dat")
        if not os.path.isfile(fn):
            continue

        df_outputs = pd.read_csv(fn, sep = "|")
        df_outputs = df_outputs[(np.abs(df_outputs["zsys"]) > 0.0001) & (df_outputs["sigcol"] != 0.0)]
        records.append(df_outputs[["trans", "col", "sigcol"]])

all_measurements = pd.concat(records, ignore_index = True)

def random_effects_mean(vals, sigmas):
    vals = np.asarray(vals, dtype = float)
    sigmas = np.asarray(sigmas, dtype = float)
    var = sigmas ** 2

    w  = 1.0 / var
    mu = (w * vals).sum() / w.sum()

    Q  = (w * (vals - mu) ** 2).sum()
    df = len(vals) - 1
    c  = w.sum() - (w ** 2).sum() / w.sum()
    T2 = max(0.0, (Q - df) / c) if c > 0 else 0.0

    w_re = 1.0 / (var + T2)
    mean = (w_re * vals).sum() / w_re.sum()
    std  = np.sqrt(1.0 / w_re.sum())
    return mean, std

combined = (
    all_measurements
    .groupby("trans", sort = True)
    .apply(lambda g: random_effects_mean(g["col"], g["sigcol"]))
    .apply(pd.Series)
    .reset_index()
    .rename(columns = {0: "col_combined", 1: "sigcol_combined"})
)

combined = combined.sort_values("trans").reset_index(drop = True)

for _, row in combined.iterrows():
    print(f"{row.trans}  {row.col_combined:.3f} ± {row.sigcol_combined:.3f}")

output_file = os.path.join(target_directory, "abundances.txt")

with open(output_file, "w") as f:
    for _, row in combined.iterrows():
        f.write(f"{row.trans} {row.col_combined:.3f} {row.sigcol_combined:.3f}\n")

CrII  13.508 ± 0.027
CuII  12.724 ± 0.099
FeII  15.245 ± 0.037
MgII  16.124 ± 0.026
NiII  13.828 ± 0.019
O I  17.000 ± 1.037
S II  15.694 ± 0.027
SiII  15.949 ± 0.050
ZnII  13.449 ± 0.031


/var/folders/7n/v6gcxcpj68q6nnv2znnc85xc0000gn/T/ipykernel_4920/817733302.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: random_effects_mean(g["col"], g["sigcol"]))


In [11]:
records = []

for i in range(len(spectra_directories)):

    directory = spectra_directories[i]
    path = target_directory / directory
    
    for subdir in os.listdir(path):
        
        subdir_path = path / subdir
        if not subdir_path.is_dir():
            continue
        fn = os.path.join(subdir_path, "compiledVPoutputs.dat")
        if not os.path.isfile(fn):
            continue

        df_SII = pd.read_csv(fn, sep = "|")
        df_SII = df_SII[df_SII["trans"].str.strip() == "S II"]
        df_SII["z_tot"] = df_SII["zsys"] + df_SII["vel"] / 299792.458
        records.append(df_SII[["z_tot", "sigvel"]])

df_SII = pd.concat(records, ignore_index = True)
bin_mw = df_SII[np.abs(df_SII["z_tot"]) <= 0.0001]
bin_sys = df_SII[np.abs(df_SII["z_tot"] - guess_redshift) <= 0.0001]
bin_mw = bin_mw[bin_mw["sigvel"] != 0]
bin_sys = bin_sys[bin_sys["sigvel"] != 0]

z_mw = random_effects_mean(bin_mw["z_tot"].values, bin_mw["sigvel"].values)[0]
z_sys = random_effects_mean(bin_sys["z_tot"].values, bin_sys["sigvel"].values)[0]
        
output_path = os.path.join(target_directory, "heliocentric_velocities.txt")
with open(output_path, "w") as f:
    f.write(f"{z_mw:.8f}\n")
    f.write(f"{z_sys:.8f}\n")       

print(f"{z_mw:.8f}")
print(f"{z_sys:.8f}")

0.00003568
0.00092797


In [ ]:
# END